# 2장 대규모 언어 모델

언어 모델의 구조를 살펴보고 TinyAgent에 모델을 연결합니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 2장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [ ]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

---

## 1. 언어 모델 연결

책의 2장에서는 언어 모델에 널리 쓰이는 트랜스포머 구조를 다룹니다. 이 실습에서는 `TinyAgent`에 언어 모델을 연결해 사용자의 질문에 답하도록 합니다.

![언어 모델을 연결한 TinyAgent](../images/ch2.png)

사용할 모델은 작업의 종류와 하드웨어 사양에 따라 달라집니다. 이 예제는 OpenAI 호환 API를 사용합니다. 요청과 응답 형식이 같은 서버라면 모델이나 서비스가 달라져도 호출 코드를 재사용할 수 있습니다. 로컬에서는 [Ollama](https://ollama.com/)나 [vLLM](https://github.com/vllm-project/vllm)을 쓸 수 있고, 클라우드에서는 해당 API 형식을 지원하는 서비스를 쓸 수 있습니다.

## 2. 추론 엔진

### 로컬 실행

공개 가중치 모델은 개인 컴퓨터에서 실행할 수 있습니다. 다음 프로그램들은 모델을 불러오고 GPU에 배치하거나 API 서버를 실행하는 작업을 지원합니다.

#### Ollama

[Ollama](https://ollama.com/)를 설치하고 [모델 목록](https://ollama.com/search)에서 사용할 모델을 내려받습니다. 첫 번째 명령은 이 장에서 프롬프트 방식에 사용하는 Gemma 3 12B를 지정합니다.

```bash
# 프롬프트로 추론과 도구 호출을 실습할 모델
ollama pull gemma3:12b

# 내장 추론과 도구 호출을 실습할 모델
ollama pull gemma4:e4b
```

모델을 내려받으면 Ollama 화면에서 직접 대화할 수 있습니다.

![Ollama 대화 화면](../images/ollama.png)

Ollama는 REST API도 제공합니다. 이 장에서는 로컬 API 서버에 요청을 보내 `gemma4:e4b`의 응답을 읽습니다.

#### LM Studio

[LM Studio](https://lmstudio.ai/)를 설치한 뒤 검색 아이콘을 클릭합니다.

![LM Studio에서 모델 검색 메뉴 열기](../images/lmstudio1.png)

모델 이름을 검색하고 파일을 내려받습니다. 아래 예시는 Gemma 4 E4B입니다.

![검색 결과에서 Gemma 모델 내려받기](../images/lmstudio2.png)

로컬 서버를 시작한 뒤 화면에 표시된 주소를 확인합니다. 이 주소가 실습에 지정할 API 주소입니다.

![LM Studio의 로컬 서버 주소 확인](../images/lmstudio3.png)

뒤에서 내장 추론과 도구 호출을 실습하려면 Gemma 4 E4B도 준비합니다.

#### llama.cpp

터미널에서 직접 모델 파일과 서버 설정을 지정하려면 [llama.cpp](https://github.com/ggml-org/llama.cpp)를 사용할 수 있습니다. 운영체제에 맞는 명령으로 설치합니다. [설치 문서](https://github.com/ggml-org/llama.cpp/blob/master/docs/install.md)

```sh
# Windows
winget install llama.cpp

# Homebrew를 사용하는 macOS 또는 Linux
brew install llama.cpp
```

설치하면 `llama-server`도 함께 제공됩니다. 서버를 시작하기 전에 양자화된 모델 파일을 내려받습니다. 양자화는 매개변수를 낮은 정밀도로 표현해 모델 크기를 줄이는 기법입니다. 예제에서는 4비트 양자화 모델을 사용합니다. [Gemma 3 파일](https://huggingface.co/unsloth/gemma-3-12b-it-GGUF/blob/main/gemma-3-12b-it-Q4_K_M.gguf) · [Gemma 4 파일](https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/blob/main/gemma-4-E4B-it-Q4_K_M.gguf)

Gemma 3 파일이 있는 디렉터리에서 다음과 같이 서버를 시작합니다.

```sh
llama-server -m ./gemma-3-12b-it-Q4_K_M.gguf --port 8080
```

Gemma 4를 사용할 때는 파일 이름을 변경합니다.

```sh
llama-server -m ./gemma-4-E4B-it-Q4_K_M.gguf --port 8080
```

브라우저로 [로컬 서버](http://localhost:8080)에 접속해 모델을 시험할 수 있습니다.

![llama.cpp 서버의 대화 화면](../images/llamacpp.png)

### 클라우드 실행

모델을 실행할 하드웨어가 없으면 원격 API를 사용할 수 있습니다. 서비스에 따라 무료 사용량, 호출 빈도와 요금이 다릅니다. 추론과 도구 호출을 지원하는 모델에서도 추론을 끄고 프롬프트 방식을 실습할 수 있습니다.

#### Google

Gemini API는 Gemini 계열 모델과 일부 Gemma 모델을 제공합니다. 무료 등급에서 실습할 때는 호출 한도를 확인하고, 예를 들어 Flash 계열이나 지원되는 공개 가중치 모델을 선택합니다. [Gemini API 키 안내](https://ai.google.dev/gemini-api/docs/api-key)에서 **Create or view a Gemini API Key**를 누릅니다.

![Gemini API 키 관리 페이지 열기](../images/google1.png)

**Create API Key**를 선택합니다.

![API 키 생성 메뉴](../images/google2.png)

필요하다면 프로젝트를 먼저 만들고, 키를 연결할 프로젝트를 선택합니다.

![프로젝트를 선택한 뒤 API 키 생성하기](../images/google3.png)

생성한 키를 복사해 모델 연결 설정에 사용합니다.

#### OpenRouter

[OpenRouter](https://openrouter.ai/)에서는 하나의 OpenAI 호환 API로 여러 회사의 공개 가중치 모델과 상용 모델을 호출할 수 있습니다. 일부 모델은 호출 한도 내에서 무료이고 다른 모델은 토큰 사용량에 따라 과금됩니다. [API 키 관리](https://openrouter.ai/keys)

#### OpenAI

OpenAI 모델을 사용하려면 해당 서비스의 API 키와 API 사용 요금이 적용됩니다. [API 키 관리](https://platform.openai.com/api-keys)

#### Claude

저자들은 Claude의 체험 크레딧으로 전체 예제를 실행할 수 있는지 확인하지 않았습니다. 계정에 적용되는 조건은 서비스의 요금 문서에서 확인할 수 있습니다. [Claude 요금 안내](https://platform.claude.com/docs/en/about-claude/pricing)

#### Cohere

Cohere의 Command 계열은 다국어와 도구 사용을 지원합니다. OpenAI 호환 엔드포인트와 체험 계정으로 실습할 수 있습니다. [Cohere](https://cohere.com/) · [API 키 관리](https://dashboard.cohere.com/api-keys)

## 3. 모델 선택

이 책은 Gemma 3에 프롬프트로 추론과 도구 호출을 지시하는 방식과, 해당 기능을 학습한 Gemma 4를 비교합니다.

![두 모델의 추론 및 도구 호출 방식](../images/evolution.png)

Gemma 3의 크기는 1B, 4B, 12B, 27B입니다. B는 10억 개의 매개변수를 뜻합니다. 저자들은 4비트로 양자화한 12B 모델을 약 12GB의 메모리 또는 비디오 메모리가 있는 환경에서 사용했습니다. 더 큰 모델로는 [27B](https://ollama.com/library/gemma3:27b)가 있습니다.

Gemma 4에는 E2B, E4B, 26B A4B, 31B가 있습니다. 저자들은 4비트 E4B 모델도 약 12GB 메모리 환경에서 시험했습니다. 더 큰 모델로는 [26B A4B](https://ollama.com/library/gemma4:26b)가 있습니다.

실습에 필요한 기능과 OpenAI 호환 엔드포인트를 제공하는 다른 모델도 사용할 수 있습니다. 저자들이 2026년 4월 집필 당시 예로 든 모델은 Gemma 4, Qwen 3.5, GPT-OSS, GLM-4.7 Flash, Phi-4입니다. 더 큰 공개 가중치 모델로는 Kimi-K2와 DeepSeek V3.2가 있습니다. 저자들이 이 모델을 모두 시험한 것은 아닙니다.

## 4. 언어 모델 호출을 담당하는 클래스

OpenAI 호환 API는 요청 매개변수와 응답 필드의 구조가 정해져 있습니다. Gemma 4를 호출해 답변과 함께 `reasoning`, `tool_calls` 같은 필드가 어디에 들어가는지 살펴봅니다. `messages`에는 여러 차례의 대화를 순서대로 담습니다. 사용자 메시지의 역할은 `user`, 모델 답변의 역할은 `assistant`입니다. 먼저 `user` 메시지 하나를 보냅니다.

In [ ]:
import json
import os
import urllib.request

from rich import print

# 요청 준비
data = json.dumps(
    {
        "model": "gemma4:e4b",
        "messages": [{"role": "user", "content": "Hi! How's life?"}]
    }
).encode("utf-8")

# 로컬 Ollama에 POST 요청
req = urllib.request.Request(
    url="http://localhost:11434/v1/chat/completions",
    data=data,
    headers={"Content-Type": "application/json"}
)

## OpenRouter를 사용하는 경우 요청 구조는 같고 URL과
## 서비스의 모델 이름, Authorization 헤더의 Bearer 토큰이 달라짐
# data = json.dumps(
#     {
#         "model": "google/gemma-4-31b-it",
#         "messages": [{"role": "user", "content": "Hi! How's life?"}]
#     }
# ).encode("utf-8")
# req = urllib.request.Request(
#     url="https://openrouter.ai/api/v1/chat/completions",
#     data=data,
#     headers={
#         "Content-Type": "application/json",
#         "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
#     },
# )

# 응답 해석
with urllib.request.urlopen(req) as response:
    result = json.loads(response.read())

print(result["choices"][0])

응답 메시지에서 확인할 필드는 다음과 같습니다.

- `content`는 모델의 답변입니다.
- `reasoning`은 모델이 제공하는 추론 내용입니다. 3장에서 살펴봅니다.
- `tool_calls`는 모델이 요청한 도구 호출입니다. 이번 요청처럼 도구를 호출하지 않으면 이 필드가 없을 수 있습니다. 5장에서 다룹니다.

`result["choices"][0]`의 `message`를 열면 답변과 추론, 도구 호출 여부를 확인할 수 있습니다. 뒤의 실습에서는 먼저 프롬프트로 추론과 도구 호출 형식을 직접 지정합니다. 그러면 프로그램이 모델 출력을 어떻게 해석하는지 확인할 수 있고, 전용 필드를 지원하지 않는 모델에도 같은 기법을 적용할 수 있습니다. 이후 모델이 제공하는 전용 필드를 사용하는 방법과 비교합니다.

모델 서버마다 응답 형식이 조금씩 다릅니다. `Response` 데이터 클래스로 답변(`content`), 추론(`reasoning`), 도구 호출(`tool_call`), 서버가 제공하는 기타 정보(`metadata`)를 같은 형식으로 다룹니다.

In [ ]:
from dataclasses import dataclass

@dataclass
class Response:
    """모델의 응답을 필드별로 저장한다."""

    content: str = ""
    reasoning: str | None = None
    tool_call: dict | None = None
    metadata: dict | None = None

지금은 `content`만 사용합니다. 모델 자체의 추론을 끈 상태에서 프롬프트가 결과에 미치는 영향을 보려면 `LLM` 클래스에서 추론을 켜고 끌 수 있어야 합니다.

In [ ]:
import json
import urllib.request


class LLM:
    def __init__(
        self,
        model: str,
        base_url: str = "http://localhost:11434/v1",
        api_key: str = "no_key",
        think: bool = False,
        temperature: float | None = None,
    ):
        """지정한 모델로 언어 모델 클라이언트를 초기화한다."""
        self.model = model
        self.base_url = base_url
        self.api_key = api_key
        self.think = think
        self.temperature = temperature

    def generate(
        self, messages: list[dict], tools: list | None = None
    ) -> Response:
        """메시지 목록을 모델에 보내 응답을 생성한다."""
        # 요청 본문 구성
        body = {
            "model": self.model,
            "messages": messages,
        }

        # 도구와 추론 설정
        if tools:
            body["tools"] = tools
        if not self.think:
            body["reasoning_effort"] = "none"
        else:
            body["reasoning_effort"] = "medium"
        if self.temperature is not None:
            body["temperature"] = self.temperature

        # OpenAI 호환 /chat/completions 엔드포인트로 POST 요청
        request = urllib.request.Request(
            f"{self.base_url}/chat/completions",
            data=json.dumps(body).encode(),
            headers={
                "Content-Type": "application/json",
                "Authorization": f"Bearer {self.api_key}",
            },
        )
        with urllib.request.urlopen(request) as response:
            data = json.loads(response.read())

        # 메시지, 도구 호출 및 응답 부가 정보 추출
        message = data["choices"][0]["message"]
        reasoning = message.get("reasoning") or message.get("reasoning_content")
        tool_calls = message.get("tool_calls")
        tool_call = tool_calls[0] if tool_calls else None
        metadata = {
            "model": data["model"],
            "prompt_tokens": data["usage"]["prompt_tokens"],
            "completion_tokens": data["usage"]["completion_tokens"],
        }

        # Response 데이터 클래스에 결과 담기
        return Response(
            content=message.get("content"),
            reasoning=reasoning,
            tool_call=tool_call,
            metadata=metadata,
        )


도구 호출을 읽는 부분은 5장에서 추가합니다.

아래 코드 설명에서 OpenAI 호환 요청을 보내고 응답을 처리하는 부분을 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch2 import llm_annotated; llm_annotated

수정한 클래스로 모델을 호출하고 응답에서 답변만 꺼냅니다.

In [ ]:
# 수정한 클래스로 언어 모델 초기화
llm = LLM(model="gemma4:e4b", think=False)

## OpenRouter를 사용하는 경우
# import os
# llm = LLM(
#     model="google/gemma-4-31b-it",
#     base_url="https://openrouter.ai/api/v1",
#     api_key=os.environ["OPENROUTER_API_KEY"],
#     think=False,
# )

# Response 데이터 클래스 생성
response = llm.generate([{"role": "user", "content": "Hi! How's life?"}])
print(response)

위에서는 Gemma 4 E4B의 추론을 껐습니다. 이후 프롬프트 방식의 예제에는 Gemma 3 12B를 사용합니다. 더 큰 모델로 같은 실습을 할 때는 프롬프트로 유도한 추론을 관찰할 수 있도록 모델 자체의 추론을 꺼 둡니다. 이후에는 내장 도구 호출과 추론을 켜고 두 구현을 비교합니다.

## 5. 실행 기록 `Trajectory`

에이전트가 여러 차례 모델을 호출하면 어느 단계에서 잘못된 도구를 선택했는지 알아보기 어렵습니다. 이를 확인하기 위해 에이전트가 수행한 각 단계를 `Step`으로 표현합니다. 도구가 돌려준 결과는 `observation`에 넣습니다.

In [ ]:
from dataclasses import dataclass

@dataclass
class Step:
    """에이전트의 실행 기록에 들어가는 한 단계이다."""

    thought: str = ""
    action: dict | None = None
    observation: str | None = None
    answer: str | None = None
    metadata: dict | None = None

`Step`은 `Response`와 비슷하지만 `answer`와 `observation`을 구분합니다. `answer`는 현재 작업을 마칠 때의 최종 답변입니다. `observation`에는 홍학을 웹에서 검색했을 때 반환된 자료처럼 도구가 실제로 돌려준 값을 넣습니다. 도구 호출은 `action`에 기록합니다. 6장의 ReAct 반복문에서 이를 생각(THOUGHT), 행동(ACTION), 관찰(OBSERVATION)로 구분합니다.

에이전트가 최종 답변을 내기까지 수행한 단계의 기록을 실행 궤적(`Trajectory`)이라고 합니다. `initialize`로 사용자 질문에 대한 기록을 시작하고, `add`로 각 단계의 추론, 행동, 관찰, 답변과 관련 정보를 추가합니다.

In [ ]:
class Trajectory:
    """에이전트의 실행을 순서대로 기록한다."""
    def __init__(self) -> None:
        self.runs: list[dict] = []

    def initialize(self, query: str) -> None:
        """주어진 질문으로 새 실행 기록을 시작한다."""
        self.runs.append({"query": query, "steps": []})

    def add(self, response: Response, observation: str | None = None) -> None:
        """모델 응답을 한 단계로 기록하고, 관찰 결과가 있으면 함께 저장한다."""
        # THOUGHT 추가
        step = Step(
            thought=response.reasoning or "",
            metadata=response.metadata,
        )

        # ACTION과 OBSERVATION 또는 ANSWER 추가
        if observation is not None:
            step.action = response.tool_call
            step.observation = observation
        else:
            step.answer = response.content
            
        self.runs[-1]["steps"].append(step)

아래 코드 설명에서 `Trajectory`가 `Step`을 만드는 과정을 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch2 import trajectory_add_annotated; trajectory_add_annotated

## 6. `agent.py` 수정

`TinyAgent`에 언어 모델을 연결해 답변을 생성합니다.

In [ ]:
class TinyAgent:
    """교육용 에이전트의 기본 클래스를 정의한다."""

    def __init__(self, llm: LLM):
        self.llm = llm
        self.memory = None  # 4장에서 메모리 추가
        self.tools = None  # 5장에서 도구 추가
        self.planner = None  # 6장에서 계획 기능 추가

        self.trajectory = Trajectory()

    def run(self, task: str) -> str:
        """주어진 작업으로 에이전트를 실행한다."""
        self.trajectory.initialize(task)
        return self._step(task)

    def _step(self, task: str) -> str:
        """에이전트의 한 단계를 실행한다."""
        messages = [{"role": "user", "content": task}]
        response = self.llm.generate(messages)
        self.trajectory.add(response)
        return response.content

    def _execute_action(self, action: str) -> str | None:
        """도구가 수행할 동작을 실행한다."""
        # 이후 장에서 구현할 부분
        return f"Executed action: {action}"

다음 비교에서 빨간색은 삭제한 코드이고 초록색은 추가한 코드입니다.

In [ ]:
from illustrated_agents.chapters.ch2 import tinyagents_diff; tinyagents_diff

구현한 에이전트에 질문을 보냅니다.

In [ ]:
agent = TinyAgent(llm=llm)
response = agent.run("What is 2 + 2?")
print(response)

이제 에이전트가 답을 계산합니다. 아래에서 전체 실행 기록을 읽고 어떤 단계에서 답이 생성되었는지 확인합니다.

In [ ]:
from rich import print

print(agent.trajectory.runs)

도구 호출이나 추론 단계가 많아지면 실행 기록도 길어집니다. `TrajectoryViewer`는 이 내용을 HTML로 표시합니다. 셀을 실행한 뒤 각 실행과 단계를 펼쳐 위에서 출력한 기록을 확인할 수 있습니다.

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

현재 `TinyAgent`는 언어 모델을 한 번 호출합니다. 3장부터 추론, 메모리, 도구와 계획 기능을 차례로 다룹니다.

**실행 결과의 차이**　언어 모델은 같은 질문에도 다른 답을 생성할 수 있습니다. 특히 소형 모델은 예제와 다른 답을 내기도 합니다. 같은 셀을 다시 실행해 결과를 비교할 수 있습니다. 저자들의 실험에서는 Gemini 3 같은 대형 모델에서 이러한 차이가 덜 자주 나타났습니다.

---

# 구현한 내용

언어 모델에 요청을 보내고, 응답을 읽고, 호출 결과를 `TinyAgent`의 실행 기록에 저장했습니다.

In [ ]:
from illustrated_agents.chapters.ch2 import what_we_built; what_we_built

# 다음 장

[3장](../chapter03/chapter03.ipynb)에서는 프롬프트로 추론을 유도하는 방법과 모델 자체의 추론 기능을 비교합니다.